# BiLSTM Temporal Branch — City Brain
**COMP 9130 Final Project — Group 5**

This notebook builds the temporal branch of the late-fusion model:
- Input: 90-day rolling windows of weather features + daily 311 complaint frequency
- Model: 2-layer Bidirectional LSTM
- Output: 3-class risk prediction (low / medium / high)

**Owner:** Savina Cai

---
## Part 0 — Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# === Adjust DATA_DIR for your environment ===
# Colab: '/content/city-brain/data'
# Local: update to your local path
import os
DATA_DIR = '/content/city-brain/data'  # Change if running locally

df_weather = pd.read_csv(os.path.join(DATA_DIR, 'weather_vancouver.csv'))
df_311 = pd.read_csv(os.path.join(DATA_DIR, '311_service_requests.csv'), sep=';', on_bad_lines='skip')
df_pavement = pd.read_csv(os.path.join(DATA_DIR, 'pavement_condition.csv'), sep=';', on_bad_lines='skip')

print(f'Weather:   {df_weather.shape}')
print(f'311:       {df_311.shape}')
print(f'Pavement:  {df_pavement.shape}')

---
## Part 1 — Target Label Construction

Map PCI Rating → 3-class risk label:
- **Low risk (0):** VERY GOOD, GOOD
- **Medium risk (1):** FAIR
- **High risk (2):** POOR, VERY POOR

In [ ]:
RISK_MAP = {
    'VERY GOOD': 0, 'GOOD': 0,   # Low risk
    'FAIR': 1,                     # Medium risk
    'POOR': 2, 'VERY POOR': 2     # High risk
}

df_pavement = df_pavement[df_pavement['PCI Rating'] != 'NO DATA'].copy()
df_pavement['risk_label'] = df_pavement['PCI Rating'].map(RISK_MAP)

print('Risk label distribution:')
print(df_pavement['risk_label'].value_counts().sort_index().rename({0:'Low',1:'Medium',2:'High'}))
print(f'\nTotal segments: {len(df_pavement)}')

---
## Part 2 — Weather Feature Engineering

Features per day:
1. Min / Max / Mean temperature
2. Total precipitation (rain + snow)
3. Freeze-thaw indicator (min < 0 AND max > 0)
4. Rolling accumulations (7-day, 30-day)

In [ ]:
# Parse and clean weather data
df_w = df_weather.copy()
df_w['date'] = pd.to_datetime(df_w['Date/Time'])
df_w = df_w.sort_values('date').reset_index(drop=True)

# Core numeric columns
num_cols = {
    'temp_min': 'Min Temp (°C)',
    'temp_max': 'Max Temp (°C)',
    'temp_mean': 'Mean Temp (°C)',
    'precip': 'Total Precip (mm)',
    'rain': 'Total Rain (mm)',
    'snow': 'Total Snow (cm)',
}

for new_name, orig_name in num_cols.items():
    df_w[new_name] = pd.to_numeric(df_w[orig_name], errors='coerce')

# Interpolate missing values (linear, as per proposal)
for col in num_cols.keys():
    df_w[col] = df_w[col].interpolate(method='linear').bfill().ffill()

# === Engineered features ===
# 1. Freeze-thaw indicator
df_w['freeze_thaw'] = ((df_w['temp_min'] < 0) & (df_w['temp_max'] > 0)).astype(int)

# 2. Temperature range (daily swing)
df_w['temp_range'] = df_w['temp_max'] - df_w['temp_min']

# 3. Rolling accumulations
for window in [7, 30]:
    df_w[f'precip_roll_{window}d'] = df_w['precip'].rolling(window, min_periods=1).sum()
    df_w[f'freeze_thaw_roll_{window}d'] = df_w['freeze_thaw'].rolling(window, min_periods=1).sum()

# Select final weather features
WEATHER_FEATURES = [
    'temp_min', 'temp_max', 'temp_mean', 'temp_range',
    'precip', 'rain', 'snow',
    'freeze_thaw',
    'precip_roll_7d', 'precip_roll_30d',
    'freeze_thaw_roll_7d', 'freeze_thaw_roll_30d',
]

df_weather_feat = df_w[['date'] + WEATHER_FEATURES].copy()
print(f'Weather features: {df_weather_feat.shape}')
print(f'Date range: {df_weather_feat["date"].min()} → {df_weather_feat["date"].max()}')
df_weather_feat.head()

---
## Part 3 — 311 Complaint Frequency Features

Aggregate daily road-related complaint counts as a city-wide temporal signal.

**Note:** Current 311 data covers 2022-08 to 2023-09. Dates outside this range
will have complaint_count = 0. Once full 311 data is downloaded, re-run this cell.

In [ ]:
# Filter road-related complaints
ROAD_KEYWORDS = ['pothole', 'road', 'pavement', 'street', 'sidewalk',
                 'curb', 'manhole', 'sewer', 'drain']

road_mask = df_311['Service request type'].str.lower().str.contains(
    '|'.join(ROAD_KEYWORDS), na=False
)
df_311_road = df_311[road_mask].copy()
df_311_road['date'] = pd.to_datetime(
    df_311_road['Service request open timestamp'], utc=True, errors='coerce'
).dt.tz_localize(None).dt.normalize()

# Daily complaint count
daily_complaints = (
    df_311_road.groupby('date').size()
    .reindex(df_weather_feat['date'], fill_value=0)
    .rename('complaint_count')
    .reset_index()
)

# Rolling complaint features
daily_complaints['complaint_roll_7d'] = (
    daily_complaints['complaint_count'].rolling(7, min_periods=1).sum()
)
daily_complaints['complaint_roll_30d'] = (
    daily_complaints['complaint_count'].rolling(30, min_periods=1).sum()
)

COMPLAINT_FEATURES = ['complaint_count', 'complaint_roll_7d', 'complaint_roll_30d']

print(f'311 road-related complaints: {len(df_311_road)}')
print(f'Days with > 0 complaints: {(daily_complaints["complaint_count"] > 0).sum()} / {len(daily_complaints)}')
daily_complaints.head()

---
## Part 4 — Build Temporal Feature Matrix

Merge weather + complaint features into a single daily time-series,
then construct 90-day sliding windows.

In [ ]:
# Merge weather + complaint features by date
df_temporal = df_weather_feat.merge(daily_complaints, on='date', how='left')
for col in COMPLAINT_FEATURES:
    df_temporal[col] = df_temporal[col].fillna(0)

ALL_FEATURES = WEATHER_FEATURES + COMPLAINT_FEATURES
print(f'Temporal feature matrix: {df_temporal.shape}')
print(f'Features ({len(ALL_FEATURES)}): {ALL_FEATURES}')
df_temporal.head()

In [ ]:
# === Build 90-day sliding windows ===
# Since pavement data is from 2020 only, we use weather windows
# ending on dates throughout the year to create temporal context.
#
# Strategy: For each pavement segment, assign a weather window
# ending at a reference date (e.g., 2020-12-31 as the survey date).
# All segments share the same weather, but each has its own risk label.
# This is a city-wide temporal signal (not per-segment weather).

WINDOW_SIZE = 90  # days

# Normalize features
scaler = StandardScaler()
feature_values = scaler.fit_transform(df_temporal[ALL_FEATURES].values)

# Create multiple windows by sliding across the full time-series
# This gives us temporal diversity for training
windows = []
window_dates = []  # end date of each window
for i in range(WINDOW_SIZE, len(feature_values)):
    windows.append(feature_values[i - WINDOW_SIZE:i])
    window_dates.append(df_temporal.iloc[i]['date'])

windows = np.array(windows)  # shape: (num_windows, 90, num_features)
window_dates = pd.Series(window_dates)

print(f'Total sliding windows: {windows.shape[0]}')
print(f'Window shape: {windows.shape}  (num_windows, days, features)')
print(f'Date range of windows: {window_dates.iloc[0]} → {window_dates.iloc[-1]}')

In [ ]:
# === Assign windows to pavement segments ===
# Pavement survey is from 2020. We use windows from 2020 to pair
# each segment with temporal context.
#
# Approach: sample windows from 2020, replicate across segments.
# Each (segment, window) pair becomes one training sample.

# Get 2020 window indices
mask_2020 = (window_dates.dt.year == 2020)
indices_2020 = np.where(mask_2020)[0]
print(f'Windows in 2020: {len(indices_2020)}')

# Sample evenly-spaced windows from 2020 (e.g., monthly)
# to avoid massive dataset while keeping seasonal variation
SAMPLE_STEP = 30  # one window per month
sampled_indices = indices_2020[::SAMPLE_STEP]
print(f'Sampled windows: {len(sampled_indices)}')
print(f'Window end dates: {[str(window_dates.iloc[i])[:10] for i in sampled_indices]}')

# Build training dataset: each segment × each sampled window
X_list = []
y_list = []
segment_ids = []

labels = df_pavement['risk_label'].values
seg_names = (df_pavement['Road Name'] + ' | ' + 
             df_pavement['From Street'].fillna('') + ' → ' + 
             df_pavement['To Street'].fillna('')).values

for win_idx in sampled_indices:
    for seg_i in range(len(labels)):
        X_list.append(windows[win_idx])  # (90, num_features)
        y_list.append(labels[seg_i])
        segment_ids.append(seg_names[seg_i])

X_all = np.array(X_list)  # (N, 90, 15)
y_all = np.array(y_list)  # (N,)

print(f'\nFinal dataset: X={X_all.shape}, y={y_all.shape}')
print(f'Class distribution:')
for c, name in enumerate(['Low', 'Medium', 'High']):
    print(f'  {name}: {(y_all == c).sum()}')

---
## Part 5 — Train / Val / Test Split

70 / 15 / 15 split, stratified by risk class.

In [ ]:
# Stratified split: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, random_state=42, stratify=y_all
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}')
for split_name, split_y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    dist = [f'{name}={int((split_y==c).sum())}' for c, name in enumerate(['L','M','H'])]
    print(f'  {split_name}: {" ".join(dist)}')

In [ ]:
class TemporalDataset(Dataset):
    """PyTorch Dataset for temporal windows."""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 64

train_loader = DataLoader(TemporalDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TemporalDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(TemporalDataset(X_test, y_test), batch_size=BATCH_SIZE)

print(f'Batches — Train: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}')

---
## Part 6 — BiLSTM Model Definition

In [ ]:
class BiLSTMBranch(nn.Module):
    """
    2-layer Bidirectional LSTM for temporal feature sequences.
    
    Input:  (batch, seq_len=90, num_features=15)
    Output: (batch, 3) — risk class logits
    
    For fusion: call get_embedding() to get the learned representation
    before the classification head.
    """
    def __init__(self, input_dim, hidden_dim=128, num_layers=2,
                 num_classes=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional

    def get_embedding(self, x):
        """Return the temporal embedding (for fusion with other branches)."""
        lstm_out, _ = self.lstm(x)        # (batch, seq_len, hidden*2)
        last_hidden = lstm_out[:, -1, :]  # (batch, hidden*2)
        return self.dropout(last_hidden)

    def forward(self, x):
        embedding = self.get_embedding(x)  # (batch, hidden*2)
        return self.fc(embedding)           # (batch, num_classes)


INPUT_DIM = len(ALL_FEATURES)
model = BiLSTMBranch(input_dim=INPUT_DIM).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

---
## Part 7 — Training Loop

In [ ]:
# === Class weights to handle imbalance ===
class_counts = np.bincount(y_train)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights = torch.FloatTensor(class_weights).to(device)
print(f'Class weights: {class_weights}')

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, verbose=True
)

NUM_EPOCHS = 50
best_val_f1 = 0
best_model_state = None
patience_counter = 0
EARLY_STOP_PATIENCE = 10

train_losses = []
val_f1s = []

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(y_batch)
    epoch_loss /= len(train_loader.dataset)
    train_losses.append(epoch_loss)

    # --- Validate ---
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())
    val_macro_f1 = f1_score(all_labels, all_preds, average='macro')
    val_f1s.append(val_macro_f1)

    scheduler.step(val_macro_f1)

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {epoch_loss:.4f} | Val F1: {val_macro_f1:.4f} | Best: {best_val_f1:.4f}')

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

print(f'\nBest validation macro F1: {best_val_f1:.4f}')

---
## Part 8 — Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(best_model_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

print('=== Test Set Results ===')
print(classification_report(
    all_labels, all_preds,
    target_names=['Low Risk', 'Medium Risk', 'High Risk']
))

test_f1 = f1_score(all_labels, all_preds, average='macro')
print(f'Test Macro F1: {test_f1:.4f}')
print(f'(Proposal target: ≥ 0.75 for fusion model, this is unimodal baseline)')

In [ ]:
# === Training curves ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(val_f1s, label='Val Macro F1')
ax2.axhline(y=best_val_f1, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_val_f1:.4f}')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Macro F1')
ax2.set_title('Validation Macro F1')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# === Confusion matrix ===
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('BiLSTM Temporal Branch — Confusion Matrix')
plt.tight_layout()
plt.show()

---
## Part 9 — Save Model & Embedding Extractor

Save the trained model so the fusion pipeline can load it
and call `get_embedding()` to extract temporal representations.

In [ ]:
import json

SAVE_DIR = os.path.join(os.path.dirname(DATA_DIR), 'modeling', 'checkpoints')
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model weights
torch.save(best_model_state, os.path.join(SAVE_DIR, 'bilstm_temporal_best.pt'))

# Save scaler and feature config for reproducibility
config = {
    'input_dim': INPUT_DIM,
    'hidden_dim': 128,
    'num_layers': 2,
    'num_classes': 3,
    'dropout': 0.3,
    'window_size': WINDOW_SIZE,
    'features': ALL_FEATURES,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'best_val_f1': float(best_val_f1),
    'test_f1': float(test_f1),
}
with open(os.path.join(SAVE_DIR, 'bilstm_temporal_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print(f'Model saved to {SAVE_DIR}')
print(f'Files: bilstm_temporal_best.pt, bilstm_temporal_config.json')
print(f'\nTo use in fusion pipeline:')
print(f'  model = BiLSTMBranch(input_dim={INPUT_DIM})')
print(f'  model.load_state_dict(torch.load("bilstm_temporal_best.pt"))')
print(f'  embedding = model.get_embedding(x)  # shape: (batch, 256)')